# Session 6: Data Reshaping
**Python for Data Analysis Workshop | World Development Indicators**

---

## Learning Objectives

By the end of this session you will be able to:
- Understand the difference between wide and long (tidy) data formats
- Convert wide data to long with `pd.melt()`
- Convert long data to wide with `df.pivot()`
- Create summary pivot tables with `df.pivot_table()`
- Sort and rank data
- Use `reset_index()` to clean up after reshaping

---

## 1. Load the Data

In [ ]:
import pandas as pd
import numpy as np

try:
    df = pd.read_csv("WDI_clean.csv")
except FileNotFoundError:
    df = pd.read_csv("World_Dev_Indicators.csv")
    df["Region"] = df["Region"].str.strip()

print(f"Shape: {df.shape}")
df.head(3)

## 2. Wide vs. Long (Tidy) Data

Data can be stored in two main shapes:

**Wide format** — each variable is its own column; each row is one entity (e.g., one country)

| Country | GDP_2020 | GDP_2021 | GDP_2022 |
|---------|----------|----------|----------|
| India   | 2.7T     | 3.2T     | 3.4T     |

**Long (Tidy) format** — each row is one observation; variables are in a single column

| Country | Year | GDP   |
|---------|------|-------|
| India   | 2020 | 2.7T  |
| India   | 2021 | 3.2T  |
| India   | 2022 | 3.4T  |

Our WDI dataset is already in **long format** (one row per country per year).
But we will create wide versions to demonstrate reshaping.

## 3. Creating a Wide-Format Dataset

Let us pivot GDP per capita to a wide format — one row per country, one column per year.

In [ ]:
# Select a clean subset for reshaping demonstrations
gdppc_col = "GDP per capita (current US$)"

# Start with a manageable subset: 5 countries, 5 years
selected = ["United States", "China", "India", "Brazil", "Germany"]
years    = [2015, 2016, 2017, 2018, 2019, 2020]

df_sub = df[
    df["Country"].isin(selected) & df["Year"].isin(years)
][["Country","Year", gdppc_col]].copy()

print("Long format (current shape):")
df_sub

In [ ]:
# Convert to wide format: rows = countries, columns = years
df_wide = df_sub.pivot(index="Country", columns="Year", values=gdppc_col)
df_wide = df_wide.round(0)
print("Wide format (countries as rows, years as columns):")
df_wide

## 4. Melting: Wide → Long with `pd.melt()`

`pd.melt()` is the reverse of pivot — it converts wide format back to long (tidy) format.
This is one of the most important reshaping operations in pandas.

In [ ]:
# Reset index so Country becomes a regular column again
df_wide_reset = df_wide.reset_index()
df_wide_reset.columns.name = None   # remove the "Year" label from columns header
print("Wide format (with reset index):")
df_wide_reset

In [ ]:
# Melt: all year columns → two new columns: variable (Year) and value (GDP per capita)
df_long = pd.melt(
    df_wide_reset,
    id_vars="Country",          # columns to keep as-is
    var_name="Year",            # name for the new column that holds old column names
    value_name=gdppc_col        # name for the new column that holds the values
)
df_long["Year"] = df_long["Year"].astype(int)   # convert Year from string to int
df_long = df_long.sort_values(["Country","Year"]).reset_index(drop=True)
print(f"Long format: {df_long.shape}")
df_long

### 4.1 Melting Multiple Value Columns

Often we have several metric columns. `melt()` handles multiple value columns too.

In [ ]:
# Create a multi-metric wide dataset for 3 countries
cols = ["Country","Year","GDP per capita (current US$)",
        "Life expectancy at birth, total (years)",
        "Population, total"]
df_multi = df[df["Country"].isin(["India","Germany","Brazil"]) & (df["Year"] >= 2018)][cols].copy()
print("Multi-metric long format (current):")
df_multi.head(6)

In [ ]:
# Melt all metric columns
df_melted = pd.melt(
    df_multi,
    id_vars=["Country","Year"],
    var_name="Indicator",
    value_name="Value"
)
df_melted = df_melted.sort_values(["Country","Year","Indicator"]).reset_index(drop=True)
print(f"Melted shape: {df_melted.shape}")
df_melted.head(12)

## 5. Pivoting: Long → Wide with `df.pivot()`

`df.pivot()` is the opposite of melt — useful when you need one row per entity
with separate columns for each time period or category.

In [ ]:
# Pivot: convert from long → wide on GDP per capita by Year
# Using the full dataset for a larger view
df_pivot = df[["Country","Year",gdppc_col]].pivot(
    index="Country",
    columns="Year",
    values=gdppc_col
)
print(f"Pivot shape: {df_pivot.shape}")
print("\nSample (first 5 countries, 5 recent years):")
df_pivot.iloc[:5, -5:]

In [ ]:
# Pivot with multiple columns requires pivot_table (handles duplicates)
# Let us create a cross-tab: average GDP per capita by Region and Income Group
df_region_income = df.pivot_table(
    index="Region",
    columns="Income Group",
    values=gdppc_col,
    aggfunc="median"
).round(0)
print("Median GDP per capita by Region and Income Group:")
df_region_income

## 6. Pivot Tables

`pivot_table()` is a powerful aggregation-plus-reshape tool — like Excel pivot tables.

In [ ]:
# Average life expectancy by Region and Decade
df["Decade"] = (df["Year"] // 10) * 10

pivot_le = df.pivot_table(
    index="Region",
    columns="Decade",
    values="Life expectancy at birth, total (years)",
    aggfunc="mean"
).round(1)

print("Average Life Expectancy by Region and Decade:")
pivot_le

In [ ]:
# Multi-index pivot: aggregate multiple metrics at once
pivot_multi = df.pivot_table(
    index="Income Group",
    values=["GDP per capita (current US$)","Life expectancy at birth, total (years)"],
    aggfunc={"GDP per capita (current US$)": "median",
             "Life expectancy at birth, total (years)": "mean"}
).round(1)
pivot_multi

In [ ]:
# Add margins (row and column totals)
pivot_with_totals = df.pivot_table(
    index="Region",
    columns="Income Group",
    values=gdppc_col,
    aggfunc="count",
    margins=True,
    margins_name="Total"
)
print("Count of observations by Region x Income Group:")
pivot_with_totals

## 7. Sorting and Ranking

In [ ]:
# Sort by a single column
df_2020 = df[df["Year"] == 2020].copy()
df_sorted = df_2020.sort_values(gdppc_col, ascending=False)
print("Top 10 by GDP per capita in 2020:")
df_sorted[["Country","Region",gdppc_col]].head(10)

In [ ]:
# Sort by multiple columns
df_multi_sort = df_2020.sort_values(
    ["Region", gdppc_col],
    ascending=[True, False]   # Region ascending, GDP descending
)
print("Sorted by Region then GDP per capita (desc):")
df_multi_sort[["Country","Region",gdppc_col]].head(10)

In [ ]:
# Rank countries within each year
df["GDP_rank"] = df.groupby("Year")[gdppc_col].rank(
    ascending=False, method="dense", na_option="bottom"
)
print("GDP per capita rank (1 = highest) in 2020:")
df[df["Year"] == 2020].nsmallest(10, "GDP_rank")[["Country","Year",gdppc_col,"GDP_rank"]]

## 8. `reset_index()` — Cleaning Up After Reshaping

After `pivot()`, the index contains meaningful data (like country names).
`reset_index()` moves that index back into regular columns.

In [ ]:
# Demonstrate reset_index
df_pivot_demo = df[["Country","Year",gdppc_col]].pivot(index="Country", columns="Year", values=gdppc_col)
print("After pivot — index type:", type(df_pivot_demo.index))
print("Index name:", df_pivot_demo.index.name)

# Reset to make Country a regular column
df_pivot_reset = df_pivot_demo.reset_index()
df_pivot_reset.columns.name = None
print("\nAfter reset_index:")
df_pivot_reset.head(3)

## 9. Summary Cheat Sheet

| Task | Code |
|------|------|
| Long → Wide | `df.pivot(index, columns, values)` |
| Wide → Long | `pd.melt(df, id_vars, var_name, value_name)` |
| Aggregated pivot table | `df.pivot_table(index, columns, values, aggfunc)` |
| Sort ascending | `df.sort_values('col')` |
| Sort descending | `df.sort_values('col', ascending=False)` |
| Sort multiple columns | `df.sort_values(['col1','col2'], ascending=[True,False])` |
| Rank | `df['col'].rank(ascending=False)` |
| Clean pivot index | `df.reset_index()` |

---

## 10. Exercises

1. Pivot the data so each row is a Region and each column is a Year. The cell values should be the **median Life Expectancy**.
2. Melt that pivot table back to long format. What does each row represent?
3. Create a pivot table showing the **count** of countries per `Income Group` per `Year` (just years 2015–2020).
4. **Bonus:** Sort countries by GDP per capita in 2020 and add a `Rank` column. Then find all countries that were in the top 20 in both 2015 and 2020.

In [ ]:
# Your answers here:
# Exercise 1

# Exercise 2

# Exercise 3

# Exercise 4 (bonus)
